# CS604 Colab launcher — Ana: which errors does kNN re-ranking fix? (no training)

kNN re-ranking gained +1.29 points on the all-class holdout (0.778 → 0.791), but the saved results do not say which errors it fixed. This reruns `scripts/rerank_knn.py` on the same checkpoint (`lora_20261026_123804`) with the current script, which also writes `rerank_changes.csv` (one row per photo: wrong before, fixed or broken). `scripts/causes_vs_rerank.py` then joins it with the error causes.

For each cause (sibling, unseen look, two looks, messy class): how many errors have the flag, how many have the truth in the top 5 (the re-ranker cannot fix the others), and how many it fixed. Photos it broke were correct before and have no per-photo flags, so breaks are only split by messy class.

**Before running:** on Drive, `CS604_RUNS/ablation_allclass_holdout_lora_no_extras/runs/A_all_classes_144min/lora_20261026_123804/` (with `last.pt` and `config.yaml`) and `CS604_RUNS/allclass_holdout_analysis/causes/` (`causes_per_error.csv`, `classes.csv`). Step 3 checks them.

Outputs in `CS604_RUNS/allclass_holdout_analysis/rerank_by_cause/`: `causes_vs_rerank.json`, `rerank_changes.csv`, `rerank.json`, `rerank_grid.csv`, `fixes_by_distance.json`. The earlier `allclass_holdout_analysis/rerank/` is left untouched. About 45–60 min on an A100: ~25 min data download, ~15 min embedding 450,000 photos, then the re-ranking grid. Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
# The all-class holdout run (colab_lora-r24_336px_all-10000-classes_topk-and-knn.ipynb) and its error analysis
RUN_DIR = f"{DRIVE_OUT}/ablation_allclass_holdout_lora_no_extras/runs/A_all_classes_144min/lora_20261026_123804"
ERRORS_CSV = f"{DRIVE_OUT}/allclass_holdout_analysis/errors/errors_lora_all.csv"   # optional: checks the split
CAUSES_DIR = f"{DRIVE_OUT}/allclass_holdout_analysis/causes"
OUT_NAME = "allclass_holdout_analysis/rerank_by_cause"
# End the runtime after the last cell, or as soon as a step fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])
assert (Path(RUN_DIR) / "last.pt").is_file() and (Path(RUN_DIR) / "config.yaml").is_file(), (
    f"no last.pt / config.yaml in {RUN_DIR}: upload the folder ablation_allclass_holdout_lora_no_extras "
    "to CS604_RUNS (keep its name), or fix RUN_DIR")
print("checkpoint found:", RUN_DIR)
for name in ("causes_per_error.csv", "classes.csv"):
    assert (Path(CAUSES_DIR) / name).is_file(), f"no {name} in {CAUSES_DIR}: run colab_error-causes_all-10000-classes.ipynb first"
print("causes found:", CAUSES_DIR)

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Rebuild the all-class split, re-rank every holdout photo and join with the causes

In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from scripts.pilot_sweep import build_pilot_root

# The same split the all-class run trained on: all 10,000 classes, 5 holdout photos each, seed 0
PILOT_ROOT, info = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 10000, 5, 0)
print(info)
if Path(ERRORS_CSV).is_file():
    holdout = {line.split()[0] for line in (Path(PILOT_ROOT) / "val.txt").read_text().splitlines() if line.strip()}
    with open(ERRORS_CSV, newline="") as f:
        err_paths = [row["path"] for row in csv.DictReader(f)]
    outside = sum(p not in holdout for p in err_paths)
    assert outside == 0, f"{outside} of {len(err_paths)} error photos are not in the rebuilt holdout: wrong split"
    print(f"split check: all {len(err_paths):,} error photos of the all-class run are in the rebuilt holdout")
else:
    print("no errors file to check the split against; continuing")
OUT = Path(DRIVE_OUT) / OUT_NAME
proc = subprocess.run([sys.executable, "-u", "scripts/rerank_knn.py", "--run_dir", RUN_DIR,
                       "--pilot_root", str(PILOT_ROOT), "--out_dir", str(OUT)])
if proc.returncode != 0:
    end_runtime(f"re-ranking failed (exit {proc.returncode})")
    raise SystemExit("re-ranking failed")
proc = subprocess.run([sys.executable, "-u", "scripts/causes_vs_rerank.py", "--changes", str(OUT / "rerank_changes.csv"),
                       "--causes_dir", CAUSES_DIR, "--out", str(OUT / "causes_vs_rerank.json")])
if proc.returncode != 0:
    end_runtime(f"joining with the causes failed (exit {proc.returncode})")
    raise SystemExit("joining with the causes failed")

## 6. Results

- **fix_rate:** share of the errors with that flag that re-ranking fixed.
- **fix_rate_when_truth_in_top5:** the same among errors it could fix at all. Compare causes on this too: a cause can have a low fix rate only because its truth is rarely in the top 5.
- **held_out_gain_points** in `rerank.json` should be about the earlier +1.29.

In [ ]:
import pandas as pd

summary = json.loads((OUT / "causes_vs_rerank.json").read_text())
print({k: summary[k] for k in ("errors_before", "fixed", "broken", "net_points_on_all_photos")})
print("held-out gain:", json.loads((OUT / "rerank.json").read_text())["held_out_gain_points"])
display(pd.DataFrame(summary["by_flag"]).T)

## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")